# Demand Forecasting
## Using Histogram Binning with Gradient Boosting Regressor

---

In [ ]:
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import sys


In [25]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import combine_raw_data

PROCESSED_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "combined_demand.csv"

combine_raw_data()
df = pd.read_csv(PROCESSED_CSV_PATH, parse_dates=["SETTLEMENT_DATE"])

c:\Users\roodruh\Desktop\repos\neso-demand-forecasting\src\data_processing.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"])


Successfully combined 5 files into: C:\Users\roodruh\Desktop\repos\neso-demand-forecasting\data\processed\combined_demand.csv


In [78]:
df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"])

df = df.sort_values(by = ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]).reset_index(drop=True)

uk_holidays = holidays.UK(years=[2021, 2022, 2023, 2024, 2025])

df["IS_HOLIDAY"] = (
    df["SETTLEMENT_DATE"].dt.date.isin(uk_holidays).astype(int)
)

df["MONTH"] = df["SETTLEMENT_DATE"].dt.month
df["DAY_OF_WEEK"] = df["SETTLEMENT_DATE"].dt.dayofweek
df["DAY_OF_YEAR"] = df["SETTLEMENT_DATE"].dt.dayofyear


df["ND_LAG_48"] = df["ND"].shift(48)
df["ND_LAG_96"] = df["ND"].shift(96)
df["ND_LAG_336"] = df["ND"].shift(336)

df["ND_AVG_24H"] = df["ND"].shift(48).rolling(window=48).mean()
df["ND_AVG_7D"] = df["ND"].shift(48).rolling(window=336).mean()

df = df.dropna().reset_index(drop=True)

In [79]:
feature_cols = [
    "ND_LAG_48",
    "ND_LAG_96",
    "ND_LAG_336",
    "ND_AVG_24H",
    "ND_AVG_7D",
    "SETTLEMENT_PERIOD",
    "IS_HOLIDAY",
    "MONTH",
    "DAY_OF_WEEK",
    "DAY_OF_YEAR",
]

X = df[feature_cols]
y = df["ND"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=False
)

In [80]:
model = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.03,
    max_leaf_nodes=63,
    min_samples_leaf=50,
    max_bins=255,
    l2_regularization=1.0,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [81]:
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
r2 = r2_score(y_test, preds)

print(f"MAE:   {mae:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAPE:  {mape:.2f}%")
print(f"R²:    {r2:.4f}")

MAE:   1310.41
RMSE:  1732.59
MAPE:  5.37%
R²:    0.9038


In [82]:
# checking which features are important
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

importances = pd.Series(result.importances_mean, index=X_train.columns)
print(importances.sort_values(ascending=False))

ND_LAG_48            0.587695
DAY_OF_WEEK          0.106278
ND_LAG_336           0.086408
SETTLEMENT_PERIOD    0.074039
DAY_OF_YEAR          0.049637
ND_LAG_96            0.026268
ND_AVG_7D            0.002217
ND_AVG_24H           0.002179
IS_HOLIDAY           0.001013
MONTH               -0.000238
dtype: float64
